# HindiMix-Noisy — GPU Baselines (Colab)
Trains **mBERT, XLM-R, MuRIL** on all noise levels.  
All results + checkpoints saved to **Google Drive** → auto-syncs to your PC.

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/hindiMix-noisy'
os.makedirs(f'{DRIVE_PROJECT}/results/tables', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results/figures', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/mbert', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/xlmr', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/muril', exist_ok=True)
print('Drive mounted. Project folder:', DRIVE_PROJECT)

## Step 2: Upload data/final/ from your PC
Run this cell — it opens a file picker. Select ALL files from your local `data/final/` folder.

In [ ]:
from google.colab import files
import os

DATA_DIR = '/content/data/final'
os.makedirs(DATA_DIR, exist_ok=True)

print('Select all CSV files from your local data/final/ folder...')
uploaded = files.upload()

import shutil
for fname, content in uploaded.items():
    dest = f'{DATA_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest}')

print('\nFiles in data/final/:')
for f in sorted(os.listdir(DATA_DIR)):
    lines = sum(1 for _ in open(f'{DATA_DIR}/{f}')) - 1
    print(f'  {f}: {lines:,} rows')

## Step 3: Install dependencies

In [ ]:
!pip install -q transformers==4.40.0 accelerate sentencepiece
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

## Step 4: Shared training code

In [ ]:
import os, json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/data/final'
DRIVE_PROJECT = '/content/drive/MyDrive/hindiMix-noisy'

class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


def load_data(noise_level):
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text','label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text','label'])
    if noise_level != 'all':
        train_df = train_df[train_df['noise_level'].isin(['clean', noise_level])]
    test_path = f'{DATA_DIR}/test_clean.csv' if noise_level == 'clean' else f'{DATA_DIR}/test_noisy_{noise_level}.csv'
    if not os.path.exists(test_path):
        test_path = f'{DATA_DIR}/test_clean.csv'
    test_df = pd.read_csv(test_path).dropna(subset=['text','label'])
    return train_df, val_df, test_df


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='  Train', leave=False):
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE)
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
    return total_loss / len(loader)


def eval_epoch(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  Eval', leave=False):
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
            targets.extend(batch['labels'].tolist())
    return f1_score(targets, preds, average='macro'), preds, targets


def train_model(model_name, short_name, noise_level, epochs=5, batch_size=32, lr=2e-5):
    print(f'\n{"="*60}')
    print(f'Model: {short_name} | Noise: {noise_level} | Device: {DEVICE}')
    print(f'{"="*60}')

    train_df, val_df, test_df = load_data(noise_level)
    print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)

    train_ds = HateSpeechDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
    val_ds   = HateSpeechDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer)
    test_ds  = HateSpeechDataset(test_df['text'].tolist(),  test_df['label'].tolist(),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

    ckpt_dir = f'{DRIVE_PROJECT}/checkpoints/{short_name}/best_{noise_level}'
    best_val_f1 = 0

    for epoch in range(epochs):
        print(f'\nEpoch {epoch+1}/{epochs}')
        loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_f1, _, _ = eval_epoch(model, val_loader)
        print(f'  Loss: {loss:.4f} | Val F1: {val_f1:.4f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            print(f'  ✓ Best saved to Drive (Val F1: {best_val_f1:.4f})')

    # Test on best model
    model = AutoModelForSequenceClassification.from_pretrained(ckpt_dir).to(DEVICE)
    test_f1, test_preds, test_targets = eval_epoch(model, test_loader)
    print(f'\nTest F1 (macro): {test_f1:.4f}')
    print(classification_report(test_targets, test_preds, target_names=['Non-hate','Hate']))

    result = {
        'model': short_name,
        'noise_level': noise_level,
        'test_f1_macro': round(test_f1, 4),
        'best_val_f1': round(best_val_f1, 4),
        'epochs': epochs, 'lr': lr
    }
    out = f'{DRIVE_PROJECT}/results/tables/{short_name}_{noise_level}.json'
    with open(out, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Results saved → Drive: {out}')
    return result

print('Shared code ready.')

## Step 5a: Train mBERT (all noise levels)

In [ ]:
mbert_results = []
for noise in ['clean', 'low', 'medium', 'high']:
    r = train_model('bert-base-multilingual-cased', 'mbert', noise, epochs=5, batch_size=32)
    mbert_results.append(r)

print('\n── mBERT Summary ──')
for r in mbert_results:
    print(f"  {r['noise_level']:8s} | val F1: {r['best_val_f1']:.4f} | test F1: {r['test_f1_macro']:.4f}")

## Step 5b: Train XLM-R (all noise levels)

In [ ]:
xlmr_results = []
for noise in ['clean', 'low', 'medium', 'high']:
    r = train_model('xlm-roberta-base', 'xlmr', noise, epochs=5, batch_size=32)
    xlmr_results.append(r)

print('\n── XLM-R Summary ──')
for r in xlmr_results:
    print(f"  {r['noise_level']:8s} | val F1: {r['best_val_f1']:.4f} | test F1: {r['test_f1_macro']:.4f}")

## Step 5c: Train MuRIL (all noise levels)
Best model for Hindi-English code-mixed text.

In [ ]:
muril_results = []
for noise in ['clean', 'low', 'medium', 'high']:
    r = train_model('google/muril-base-cased', 'muril', noise, epochs=5, batch_size=32)
    muril_results.append(r)

print('\n── MuRIL Summary ──')
for r in muril_results:
    print(f"  {r['noise_level']:8s} | val F1: {r['best_val_f1']:.4f} | test F1: {r['test_f1_macro']:.4f}")

## Step 6: Full results table (all models)

In [ ]:
import pandas as pd, json, os

rows = []
for f in sorted(os.listdir(f'{DRIVE_PROJECT}/results/tables')):
    if f.endswith('.json'):
        with open(f'{DRIVE_PROJECT}/results/tables/{f}') as fp:
            rows.append(json.load(fp))

results_df = pd.DataFrame(rows)
pivot = results_df.pivot(index='model', columns='noise_level', values='test_f1_macro')
pivot = pivot[['clean','low','medium','high']]
pivot['degradation'] = pivot['clean'] - pivot['high']
print(pivot.round(4).to_string())

# Save master table to Drive
pivot.to_csv(f'{DRIVE_PROJECT}/results/tables/ALL_RESULTS.csv')
print('\nSaved ALL_RESULTS.csv to Drive')

## Step 7: Download all results to your PC
This downloads the results folder as a zip. Also already saved to Drive for auto-sync.

In [ ]:
import shutil
from google.colab import files

# Zip the results/tables folder
shutil.make_archive('/content/results_tables', 'zip', f'{DRIVE_PROJECT}/results/tables')
files.download('/content/results_tables.zip')
print('Downloading results_tables.zip ...')
print('Extract to: data/results/tables/ in your local project')